# 04 · Document Loader：把各种格式读进来

> RAG 第一步：把异构文件统一读成带来源的“文本”。别小看这步，很多生产 RAG 的问题出在解析，而不是模型。

**本文件覆盖知识点**：PDF / Word / Markdown / HTML / TXT / Excel / CSV / PPT / JSON / 数据库 / 网页 / API / 图片；工具：PyPDF·PyMuPDF·Unstructured·Docling·MinerU·OCR

In [ ]:
# ===== 本课共用：真调 LLM 做「说明 / 演示」的小助手 =====
# 凡某个知识点能靠“真调一次大模型”当场讲清 / 演示的，下面的 cell 都用 _llm_live()
# 真调 qwen-plus 并打印模型输出作为说明；只有在项目根 .env 配了 DASHSCOPE_API_KEY 时才真调，
# 没配置就打印一段固定的演示样例，保证整个 notebook 不联网也能完整读下来。
from dotenv import load_dotenv; load_dotenv()
import os
from dashscope import Generation

_KEY = os.getenv('DASHSCOPE_API_KEY', '').strip()
_HAS_KEY = bool(_KEY) and '你的' not in _KEY

def _llm_live(prompt, fallback, system='你是资深 RAG 讲师，回答精炼、结构清晰、尽量结合例子。', temperature=0.3, model='qwen-plus'):
    """真调一次 qwen-plus 并打印结果；无 Key 时打印 fallback 作为演示样例。返回模型文本或 None。"""
    if not _HAS_KEY:
        print('未在 .env 配置 DASHSCOPE_API_KEY，跳过实时调用。以下是固定演示样例（配置后自动变为实时输出）：')
        print(fallback)
        return None
    msgs = [{'role': 'system', 'content': system}, {'role': 'user', 'content': prompt}]
    try:
        r = Generation.call(model=model, messages=msgs, temperature=temperature, result_format='message', api_key=_KEY)
        if r.status_code == 200:
            text = r.output.choices[0].message.content
            print('—— 模型实时输出 ——')
            print(text)
            return text
        print('调用失败：', getattr(r, 'code', ''), getattr(r, 'message', ''))
    except Exception as e:
        print('调用异常：', e)
    print('fallback：')
    print(fallback)
    return None


## 1. 支持的格式地图

| 格式 | 本质 | 读取难度 | 常见工具 |
|------|------|---------|---------|
| TXT / Markdown | 纯文本 | ★ | 直接 open |
| CSV / JSON | 结构化文本 | ★ | csv / json |
| HTML | 带标签文本 | ★★ | BeautifulSoup |
| PDF | 二进制排版 | ★★★ | PyMuPDF / PDFPlumber |
| Word / PPT | 二进制+结构 | ★★★ | python-docx / python-pptx |
| Excel | 表格 | ★★ | openpyxl / pandas |
| 网页 / API | 动态获取 | ★★ | requests + 反爬处理 |
| 图片 / 扫描件 | 像素 | ★★★★ | OCR（见下） |

生产级解析方案（更全更稳）：Unstructured、Docling（IBM）、MinerU（书生，中文 PDF 强），以及各类 OCR 引擎。

In [1]:
# 从 txt/md 开始：直接读，并保留来源元信息(metadata)
from pathlib import Path
import os

def load_text_file(path: Path) -> dict:
    """读取 txt/md，返回 {'content':正文, 'metadata':来源信息}"""
    with open(path, 'r', encoding='utf-8') as f:
        return {
            'content': f.read(),
            'metadata': {'source': path.name, 'filetype': path.suffix},
        }

sample = load_text_file(Path('data/星云智能产品手册.md'))
print('来源:', sample['metadata']['source'])
print('正文长度:', len(sample['content']), '字符')
print('开头:', sample['content'][:80], '...')

来源: 星云智能产品手册.md
正文长度: 398 字符
开头: # 星云智能客服机器人产品手册

星云智能客服机器人是星云智能科技公司推出的一款基于大模型的智能客服产品，支持 7x24 小时自动应答客户咨询。

## 产品功 ...


## 2. 统一入口设计

生产里往往一个 `data/` 目录混着多种格式。分派器（dispatcher）按扩展名自动挑对应的加载器：

```text
data/xxx.md   → load_text_file   (本课)
data/xxx.pdf  → PDF 加载器        (第 05 课演示)
data/xxx.docx → docx 加载器        (扩展点)
```

## 3. 分派器：按扩展名自动选择加载器


In [2]:
# 分派器：对目录内所有文件按后缀分派（PDF 等扩展加载器后续自行补充）
def load_documents(data_dir: str) -> list:
    docs = []
    for path in sorted(Path(data_dir).glob('*')):
        if not path.is_file():
            continue
        ext = path.suffix.lower()
        if ext in ('.txt', '.md', '.markdown'):
            docs.append(load_text_file(path))
        elif ext == '.pdf':
            print('PDF 解析见第 05 课，先跳过:', path.name)
        else:
            print('本课暂不处理:', path.name)
    return docs

docs = load_documents('data')
print('已加载文档数:', len(docs))
print('数据源:', [d['metadata']['source'] for d in docs])

已加载文档数: 2
数据源: ['向量数据库.md', '星云智能产品手册.md']


In [ ]:
# 知识点·真调说明：格式地图与工具选型 —— 让模型当“数据接入顾问”，按文件特征逐点选工具、点出坑
_llm_live(
    prompt='你负责把一个企业的历史资料接进 RAG 知识库，收到三种文件，请各给出“用什么方案、先做什么、会踩什么坑”：\n'
           '① 500 页中文 PDF：带目录、双栏排版，内页含大量表格，其中约 50 页是扫描件；\n'
           '② 一份含表格的 .docx + 一份老式 .doc；\n'
           '③ 一个需登录才能下载报表的内部网页。',
    system='你是资深 RAG 数据接入工程师。参考工具池：PyMuPDF/PDFPlumber、python-docx、Unstructured、'
           'Docling、MinerU、PaddleOCR/RapidOCR、BeautifulSoup/requests。逐条按“文件 → 推荐方案 → 关键坑”作答，每条不超过 2 行。',
    fallback='① → MinerU / Docling（版面+表格），扫描页走 PaddleOCR/RapidOCR，双栏要按栏还原顺序；'
             '坑：表格易被压成无空格文本、繁体/公式要专门模型。\n'
             '② → python-docx 读正文与表格，老式 .doc 先转存 .docx；坑：图片/公式/批注易丢。\n'
             '③ → requests + BeautifulSoup，先处理登录态与反爬，再抽正文；坑：导航/页脚等噪声混进正文。',
    temperature=0.2,
)
print('→ 加载不是“一行 open”，而是按 格式复杂度（文本<排版<扫描件）+ 内容类型 选工具链；'
      '选型对了，后面的解析(05)与清洗(06)才省力——这正是本课格式地图的意义。')

In [ ]:
# 知识点·真调说明：Loader 的 metadata（来源盖章） —— 片段不带“出自哪份文档”，模型就答不清归属、无法溯源
print('① 检索片段丢了文档标题（假设 Loader 没盖来源章）：功能能读，但说不清属于哪款产品')
_llm_live(
    prompt='【资料片段·文档标题与章节已丢失】产品面向客服场景，支持多轮对话、知识库问答、'
           '工单自动创建与流转；另有资料提到，面向运维工单的产品可自动分类与分派，并支持把数万条历史工单'
           '批量导入后按历史惯例“冷启动”。\n'
           '【问题】客户想把 3 万条历史工单一次性导入、让系统按历史惯例自动分派，应选哪款产品？依据出自哪份文档？',
    system='你是接入顾问。只能依据资料判断；资料里没写明的产品归属/出处，明确答“无法确认”，不要猜。',
    fallback='能从字面看出“批量导入历史工单”更贴合运维工单产品，但片段没有文档标题，无法说明依据出自哪份文档——'
             '来源在加载时丢了。',
    temperature=0.2,
)
print()
print('② 带来源元数据（Loader 按文档+章节盖章）：能定位是哪款产品、并可溯源')
_llm_live(
    prompt='【文档：《星云智能客服机器人产品手册》】产品面向客服场景，支持多轮对话、知识库问答、'
           '工单自动创建与流转，答不上来自动转人工。\n'
           '【文档：《星云智能工单机器人产品手册》】面向运维工单，自动分类与分派，支持把数万条历史工单'
           '批量导入后按历史惯例“冷启动”。\n'
           '【问题】客户想把 3 万条历史工单一次性导入、让系统按历史惯例自动分派，应选哪款产品？依据出自哪份文档？',
    system='你是接入顾问。只能依据资料判断，回答最后必须注明出自哪份文档。',
    fallback='应选《星云智能工单机器人产品手册》：它明确支持把数万条历史工单批量导入并按历史惯例冷启动；'
             '《星云智能客服机器人产品手册》的“工单自动创建与流转”面向客服而非批量导入。',
    temperature=0.2,
)
print()
print('同一批功能，是否带“文档标题/章节”的来源盖章，决定了下游能否 溯源、按文档过滤、去重——')
print('→ 这就是 Loader 产出“正文 content + 来源 metadata”而不是只给一串文本的原因；'
      '分派器(dispatcher)按扩展名自动把来源信息带上，正是为了保住这条溯源链。')

## 小结

- Loader 解决“异构格式 → 统一文本 + 来源元信息”；
- 选工具看格式复杂度：文本 < 排版 < 扫描件；
- metadata 是后面溯源和过滤的基础。

PDF 这类排版文档只读文本会丢结构（表格、图片、标题），下一课讲结构化解析。